In [ ]:
!pip install -q kagglehub

In [ ]:
import kagglehub

path = kagglehub.dataset_download("avc0706/luna16")

print("Dataset saved at:", path)

100%|██████████| 32.1G/32.1G [18:32<00:00, 31.0MB/s]

Extracting files...


Dataset saved at: /root/.cache/kagglehub/datasets/avc0706/luna16/versions/1


In [ ]:
import os

for root, dirs, files in os.walk(path):
    print(root)
    print(files[:5])
    break

/root/.cache/kagglehub/datasets/avc0706/luna16/versions/1
['annotations.csv', 'sampleSubmission.csv', 'candidates.csv']


In [ ]:
!pip install -q kaggle
!pip install -q SimpleITK
!pip install -q scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 46.9 MB/s eta 0:00:00


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

import SimpleITK as sitk

In [ ]:
def load_itk(path):
    img = sitk.ReadImage(path)
    arr = sitk.GetArrayFromImage(img)
    origin = np.array(img.GetOrigin())
    spacing = np.array(img.GetSpacing())
    return arr, origin, spacing

In [ ]:
import os

def show_tree(path, max_depth=3):
    for root, dirs, files in os.walk(path):
        level = root.replace(path, "").count(os.sep)

        if level > max_depth:
            continue

        indent = " " * 4 * level
        print(f"{indent}{os.path.basename(root)}/")

        subindent = " " * 4 * (level + 1)
        for f in files[:5]:   # chỉ hiện 5 file đầu
            print(f"{subindent}{f}")

show_tree("/root/.cache/kagglehub/datasets/avc0706/luna16/versions/1")

1/
    annotations.csv
    sampleSubmission.csv
    candidates.csv
    subset1/
        subset1/
            1.3.6.1.4.1.14519.5.2.1.6279.6001.128059192202504367870633619224.mhd
            1.3.6.1.4.1.14519.5.2.1.6279.6001.910607280658963002048724648683.raw
            1.3.6.1.4.1.14519.5.2.1.6279.6001.134370886216012873213579659366.raw
            1.3.6.1.4.1.14519.5.2.1.6279.6001.200558451375970945040979397866.mhd
            1.3.6.1.4.1.14519.5.2.1.6279.6001.152684536713461901635595118048.mhd
    evaluationScript/
        evaluationScript/
            noduleCADEvaluationLUNA16.py
            NoduleFinding.py
            exampleFiles/
            tools/
                csvTools.py
                __init__.py
            annotations/
                seriesuids.csv
                annotations.csv
                annotations_excluded.csv
    subset2/
        subset2/
            1.3.6.1.4.1.14519.5.2.1.6279.6001.220205300714852483483213840572.raw
            1.3.6.1.4.1.14519.5.2.1.627

In [ ]:
import os

for root, _, files in os.walk(path):
    if "annotations.csv" in files:
        ann_path = os.path.join(root, "annotations.csv")
        break

print("Annotation path:", ann_path)

annotations = pd.read_csv(ann_path)

Annotation path: /root/.cache/kagglehub/datasets/avc0706/luna16/versions/1/annotations.csv


In [ ]:
import os
import pandas as pd

ann_path = os.path.join(path, "/root/.cache/kagglehub/datasets/avc0706/luna16/versions/1/annotations.csv")

annotations = pd.read_csv(ann_path)
annotations.head()

,seriesuid,coordX,coordY,coordZ,diameter_mm
0,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,-128.699421,-175.319272,-298.387506,5.651471
1,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,103.783651,-211.925149,-227.121250,4.224708
2,1.3.6.1.4.1.14519.5.2.1.6279.6001.100398138793...,69.639017,-140.944586,876.374496,5.786348
3,1.3.6.1.4.1.14519.5.2.1.6279.6001.100621383016...,-24.013824,192.102405,-391.081276,8.143262
4,1.3.6.1.4.1.14519.5.2.1.6279.6001.100621383016...,2.441547,172.464881,-405.493732,18.545150


In [ ]:
def world_to_voxel(world, origin, spacing):
    stretched = np.abs(world - origin)
    voxel = stretched / spacing
    return voxel.astype(int)

In [ ]:
def extract_patch(volume, center, size=64):

    z,y,x = center
    h = size//2

    patch = volume[
        max(0,z-h):z+h,
        max(0,y-h):y+h,
        max(0,x-h):x+h
    ]

    pad = [(0,size-s) for s in patch.shape]
    patch = np.pad(patch,pad,mode='constant')

    return patch

In [ ]:
def random_negative(volume):

    z_max,y_max,x_max = volume.shape

    for _ in range(20):

        z = np.random.randint(32, z_max-32)
        y = np.random.randint(32, y_max-32)
        x = np.random.randint(32, x_max-32)

        patch = volume[z-16:z+16, y-16:y+16, x-16:x+16]

        if np.mean(patch) > -800:
            return np.array([z,y,x])

    return np.array([z_max//2,y_max//2,x_max//2])

In [ ]:
class LunaDataset(Dataset):

    def __init__(self, mhd_files, annotations):

        self.samples = []

        for f in mhd_files:

            volume, origin, spacing = load_itk(f)

            if min(volume.shape) < 64:
               continue

            uid = os.path.basename(f).replace(".mhd","")
            nodules = annotations[annotations.seriesuid==uid]


            for _,row in nodules.iterrows():

                world = np.array(
                    [row.coordZ,row.coordY,row.coordX]
                )

                voxel = world_to_voxel(world,origin,spacing)

                patch = extract_patch(volume,voxel)

                self.samples.append((patch,1))


            for _ in range(len(nodules)):

                center = random_negative(volume)
                patch = extract_patch(volume,center)

                self.samples.append((patch,0))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self,idx):

        patch,label = self.samples[idx]

        patch = patch.astype(np.float32)


        patch = np.clip(patch,-1000,400)
        patch = (patch + 1000)/1400

        patch = torch.tensor(patch).unsqueeze(0)

        return patch, torch.tensor(label,dtype=torch.float32)

In [ ]:
mhd_files = []

for root, _, files in os.walk(path):   # ⭐ dùng path từ kagglehub
    for f in files:
        if f.endswith(".mhd"):
            mhd_files.append(os.path.join(root, f))

print("Total mhd files:", len(mhd_files))
print("Example:", mhd_files[:3])

Total mhd files: 1333
Example: ['/root/.cache/kagglehub/datasets/avc0706/luna16/versions/1/subset1/subset1/1.3.6.1.4.1.14519.5.2.1.6279.6001.128059192202504367870633619224.mhd', '/root/.cache/kagglehub/datasets/avc0706/luna16/versions/1/subset1/subset1/1.3.6.1.4.1.14519.5.2.1.6279.6001.200558451375970945040979397866.mhd', '/root/.cache/kagglehub/datasets/avc0706/luna16/versions/1/subset1/subset1/1.3.6.1.4.1.14519.5.2.1.6279.6001.152684536713461901635595118048.mhd']


In [ ]:
class Nodule3DCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv3d(1,16,3,padding=1),
            nn.BatchNorm3d(16),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(16,32,3,padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(32,64,3,padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool3d(1)
        )

        self.fc = nn.Linear(64,1)

    def forward(self,x):

        x = self.features(x)
        x = x.view(x.size(0),-1)

        return self.fc(x)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = Nodule3DCNN().to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [ ]:
from sklearn.model_selection import train_test_split

train_files, val_files = train_test_split(
    mhd_files,
    test_size=0.2,
    random_state=42
)

print(len(train_files), len(val_files))

1066 267


In [ ]:
train_ds = LunaDataset(train_files, annotations)
val_ds = LunaDataset(val_files, annotations)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_ds,
    batch_size=4,
    shuffle=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=4,
    shuffle=False
)

print("Train batches:", len(train_loader))

Train batches: 721


In [ ]:
EPOCHS = 6

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for x,y in train_loader:

        x,y = x.to(device), y.to(device)

        pred = model(x).squeeze()

        loss = criterion(pred,y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    #valid
    model.eval()

    preds = []
    gts = []

    with torch.no_grad():

        for x,y in val_loader:

            x = x.to(device)

            out = model(x).squeeze()

            prob = torch.sigmoid(out).cpu().numpy()

            preds.extend(prob)
            gts.extend(y.numpy())

    preds = np.array(preds)
    gts = np.array(gts)

    threshold = 0.5
    binary = (preds > threshold).astype(int)

    auc = roc_auc_score(gts,preds)
    acc = accuracy_score(gts,binary)
    precision = precision_score(gts,binary)
    recall = recall_score(gts,binary)
    f1 = f1_score(gts,binary)

    tn,fp,fn,tp = confusion_matrix(gts,binary).ravel()
    specificity = tn/(tn+fp)

    print(f"""
Epoch {epoch}

Loss        : {total_loss:.3f}
AUC         : {auc:.4f}
Accuracy    : {acc:.4f}
Precision   : {precision:.4f}
Recall      : {recall:.4f}
F1-score    : {f1:.4f}
Specificity : {specificity:.4f}
""")


Epoch 0

Loss        : 466.320
AUC         : 0.6533
Accuracy    : 0.5847
Precision   : 0.5486
Recall      : 0.9556
F1-score    : 0.6971
Specificity : 0.2139


Epoch 1

Loss        : 463.409
AUC         : 0.6253
Accuracy    : 0.5833
Precision   : 0.5485
Recall      : 0.9417
F1-score    : 0.6933
Specificity : 0.2250


Epoch 2

Loss        : 461.991
AUC         : 0.6577
Accuracy    : 0.5903
Precision   : 0.5512
Recall      : 0.9722
F1-score    : 0.7035
Specificity : 0.2083


Epoch 3

Loss        : 462.023
AUC         : 0.6269
Accuracy    : 0.5542
Precision   : 0.6639
Recall      : 0.2194
F1-score    : 0.3299
Specificity : 0.8889


Epoch 4

Loss        : 461.098
AUC         : 0.6336
Accuracy    : 0.5875
Precision   : 0.5514
Recall      : 0.9389
F1-score    : 0.6948
Specificity : 0.2361


Epoch 5

Loss        : 458.520
AUC         : 0.6038
Accuracy    : 0.5681
Precision   : 0.5437
Recall      : 0.8472
F1-score    : 0.6623
Specificity : 0.2889

